In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn import metrics
from tensorflow import keras
from sklearn.manifold import TSNE
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
# import keras_tuner as kt

import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
import os, copy, random, pickle, datetime

# Data preprocessing

In [ ]:
file_path = './MLT_data.xlsx'

In [ ]:
df = pd.DataFrame()
for i in range(0, 100+1, 5):
    temp = pd.read_excel(file_path, sheet_name=str(i), index_col=0)
    temp = temp.loc[temp['DCIR']<100, :]
    temp.dropna(inplace=True)
    df = pd.concat([df, temp])

df['DCIR'] = [i/34.18*100 for i in df['DCIR']]
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

col = ['temp', 'DCIR', 'SoH'] + list(df.columns[3:])
df = df[col]
df

In [ ]:
feature = df.columns[3:]

mms = MinMaxScaler().fit(df[col])
data_transform = pd.DataFrame(mms.transform(df[col]), columns=col)

# MTL (Multi-Task Learning) model training

In [ ]:
X = data_transform[feature]
y1 = df['temp']
y2 = df['SoH']
y3 = df['DCIR']

X_train, X_test, y1_train, y1_test, y2_train, y2_test, y3_train, y3_test= train_test_split(X, y1, y2, y3, test_size=0.15, random_state=40)

train_dataset_raw = tf.data.Dataset.from_tensor_slices((X_train, (y1_train, y2_train, y3_train)))
test_dataset_raw = tf.data.Dataset.from_tensor_slices((X_test, (y1_test, y2_test, y3_test)))

BATCH_SIZE = 512

AUTOTUNE = tf.data.AUTOTUNE
def preprocess_fn(features, labels_tuple):
    features = tf.cast(features, tf.float32)
    return features, labels_tuple

train_dataset = (
    train_dataset_raw
    .shuffle(buffer_size=len(X_train))
    .map(preprocess_fn, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

train_dataset_plot = (
    train_dataset_raw
    .map(preprocess_fn, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_dataset = (
    test_dataset_raw
    .map(preprocess_fn, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

In [ ]:
# This code performs hyperparameter optimization (specifically, the number of hidden layers and nodes) for the MTL model.
# Note: The execution time is significant and may take several days depending on your hardware specifications.

# X_train, X_val, y1_train, y1_val, y2_train, y2_val, y3_train, y3_val= train_test_split(X_train, y1_train, y2_train, y3_train, test_size=0.15, random_state=40)
# val_dataset_raw = tf.data.Dataset.from_tensor_slices((X_val, (y1_val, y2_val, y3_val)))

# val_dataset = (
#     val_dataset_raw
#     .map(preprocess_fn, num_parallel_calls=AUTOTUNE)
#     .batch(BATCH_SIZE)
#     .prefetch(AUTOTUNE)
# )

# def build_model(hp):
#     tf.keras.backend.clear_session()

#     input_layer = keras.layers.Input(shape=(X_train.shape[1],))
#     x = input_layer

#     for i in range(hp.Int('num_layers', 2, 5)):
#         x = keras.layers.Dense(
#             units=hp.Choice(f'units_{i}', values=[16, 32, 64, 128, 256]),
#             activation='relu')(x)

#     last_hidden_layer = x

#     output_y1 = layers.Dense(1, name='output_y1')(last_hidden_layer)
#     output_y2 = layers.Dense(1, name='output_y2')(last_hidden_layer)
#     output_y3 = layers.Dense(1, name='output_y3')(last_hidden_layer)

#     lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(0.001, decay_steps=3000, decay_rate=0.96, staircase=True)
#     optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

#     model = keras.Model(inputs=input_layer, outputs=[output_y1, output_y2, output_y3])
#     model.compile(
#         optimizer=optimizer, jit_compile=True,
#         loss={'output_y1': 'mean_squared_error', 'output_y2': 'mean_squared_error', 'output_y3': 'mean_squared_error'},
#         loss_weights={'output_y1': 100, 'output_y2': 10, 'output_y3': 1},
#     )
#     return model

# early_stopping = EarlyStopping(monitor='loss', patience=100, restore_best_weights=True)


# current_path = os.getcwd() + '/my_dir'
# base_path = Path(current_path).resolve()
# os.makedirs(base_path, exist_ok=True)

# base_dir = os.path.abspath("my_dir")
# tuner = kt.BayesianOptimization(
#     hypermodel=build_model, objective='val_loss', max_trials=200, executions_per_trial=5,
#     directory= base_dir, project_name='MTL_optimization', overwrite=True,
# )

# tuner.search(train_dataset, epochs=1500, validation_data=val_dataset, batch_size=BATCH_SIZE, verbose=0, callbacks=[early_stopping])

# best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

# print("\n=======================================================")
# print(f"최적의 레이어 수: {best_hps.get('num_layers')}개")
# for i in range(best_hps.get('num_layers')):
#     print(f"  레이어 {i+1} 노드 수: {best_hps.get(f'units_{i}')}개")
# print("=======================================================")

# best_model = tuner.get_best_models(num_models=1)[0]

In [ ]:
input_layer = keras.Input(shape=(58,))

shared_dense_1_1 = layers.Dense(256, activation='relu')(input_layer)
shared_dense_1_2 = layers.Dense(256, activation='relu')(shared_dense_1_1)
shared_dense_1_3 = layers.Dense(256, activation='relu')(shared_dense_1_2)
shared_dense_1_4 = layers.Dense(256, activation='relu')(shared_dense_1_3)
shared_dense_1_5 = layers.Dense(256, activation='relu')(shared_dense_1_4)

output_y1 = layers.Dense(1, name='output_y1')(shared_dense_1_5)
output_y2 = layers.Dense(1, name='output_y2')(shared_dense_1_5)
output_y3 = layers.Dense(1, name='output_y3')(shared_dense_1_5)

best_model = keras.Model(inputs=input_layer, outputs=[output_y1, output_y2, output_y3])
best_model.compile(
    optimizer='adam',
    loss={'output_y1': 'mean_squared_error', 'output_y2': 'mean_squared_error', 'output_y3': 'mean_squared_error'},
    loss_weights={'output_y1': 100, 'output_y2': 10, 'output_y3': 1},
)

early_stopping = EarlyStopping(monitor='loss', patience=100, restore_best_weights=True)

history = best_model.fit(train_dataset, epochs=1500, batch_size=BATCH_SIZE, verbose=True, callbacks=[early_stopping])

In [ ]:
result_train = best_model.predict(train_dataset_plot)
result_test = best_model.predict(test_dataset)

plt.figure(figsize=(4,4))
plt.plot([min(df['temp'])-1, max(df['temp'])+1], [min(df['temp'])-1, max(df['temp'])+1], 'k--')
plt.scatter(y1_train, result_train[0], alpha=0.5)
plt.scatter(y1_test, result_test[0], alpha=0.5)
plt.title('Max. temp.')
plt.xlabel('True value')
plt.ylabel('Predicted value')
plt.show()
print('R2', metrics.r2_score(y1_train, result_train[0]), metrics.r2_score(y1_test, result_test[0]))
print('RMSE', metrics.root_mean_squared_error(y1_train, result_train[0]), metrics.root_mean_squared_error(y1_test, result_test[0]))
print('MAE', metrics.mean_absolute_error(y1_train, result_train[0]), metrics.mean_absolute_error(y1_test, result_test[0]))
print('MAPE', metrics.mean_absolute_percentage_error(y1_train, result_train[0])*100, metrics.mean_absolute_percentage_error(y1_test, result_test[0])*100)

plt.figure(figsize=(4,4))
plt.plot([min(df['SoH'])-3, max(df['SoH'])+3], [min(df['SoH'])-3, max(df['SoH'])+3], 'k--')
plt.scatter(y2_train, result_train[1], alpha=0.5)
plt.scatter(y2_test, result_test[1], alpha=0.5)
plt.title('SoH')
plt.xlabel('True value')
plt.ylabel('Predicted value')
plt.show()
print('R2', metrics.r2_score(y2_train, result_train[1]), metrics.r2_score(y2_test, result_test[1]))
print('RMSE', metrics.root_mean_squared_error(y2_train, result_train[1]), metrics.root_mean_squared_error(y2_test, result_test[1]))
print('MAE', metrics.mean_absolute_error(y2_train, result_train[1]), metrics.mean_absolute_error(y2_test, result_test[1]))
print('MAPE', metrics.mean_absolute_percentage_error(y2_train, result_train[1])*100, metrics.mean_absolute_percentage_error(y2_test, result_test[1])*100)

plt.figure(figsize=(4,4))
plt.plot([min(df['DCIR'])-5, max(df['DCIR'])+5], [min(df['DCIR'])-5, max(df['DCIR'])+5], 'k--')
plt.scatter(y3_train, result_train[2], alpha=0.5)
plt.scatter(y3_test, result_test[2], alpha=0.5)
plt.title('DCIR increase rate')
plt.xlabel('True value')
plt.ylabel('Predicted value')
plt.show()
print('R2', metrics.r2_score(y3_train, result_train[2]), metrics.r2_score(y3_test, result_test[2]))
print('RMSE', metrics.root_mean_squared_error(y3_train, result_train[2]), metrics.root_mean_squared_error(y3_test, result_test[2]))
print('MAE', metrics.mean_absolute_error(y3_train, result_train[2]), metrics.mean_absolute_error(y3_test, result_test[2]))
print('MAPE', metrics.mean_absolute_percentage_error(y3_train, result_train[2])*100, metrics.mean_absolute_percentage_error(y3_test, result_test[2])*100)

In [ ]:
# plt.plot(range(50), history.history['loss'])
plt.plot(range(len(history.history['loss'])), history.history['output_y1_loss'], label='Max. temp.')
plt.plot(range(len(history.history['loss'])), history.history['output_y2_loss'], label='SoH')
plt.plot(range(len(history.history['loss'])), history.history['output_y3_loss'], label='DCIR increase rate')
plt.xlabel('Epoch', fontsize=13)
plt.ylabel('Loss', fontsize=13)
plt.legend()
plt.ylim([-1, 40])
plt.show()

# IG (Integrated Gradient) value calculation

In [ ]:
def get_integrated_gradients_batch(model, input_batch, target_idx=None, baseline=None, m_steps=50):
    if hasattr(input_batch, 'values'):
        input_batch = input_batch.values

    input_batch = tf.cast(input_batch, tf.float32)

    if len(baseline.shape) == 1:
            baseline = tf.reshape(baseline, (1, len(baseline)))
    baseline = tf.cast(baseline, tf.float32)
    alphas = tf.linspace(start=0.0, stop=1.0, num=m_steps+1)[:, tf.newaxis, tf.newaxis]

    input_expanded = tf.expand_dims(input_batch, axis=0) # Shape: (1, Batch, Feat)
    baseline_expanded = tf.expand_dims(baseline, axis=0) # Shape: (1, Feat) or (1, Batch, Feat)

    interpolated = baseline_expanded + alphas * (input_expanded - baseline_expanded)
    interpolated_reshaped = tf.reshape(interpolated, (-1, input_batch.shape[1]))

    with tf.GradientTape() as tape:
        tape.watch(interpolated_reshaped)
        predictions = model(interpolated_reshaped)

        if isinstance(predictions, list):
            target_prediction = predictions[target_idx]
        else:
            target_prediction = predictions

    grads = tape.gradient(target_prediction, interpolated_reshaped)
    grads = tf.reshape(grads, (m_steps+1, len(input_batch), -1))
    avg_grads = tf.reduce_mean(grads, axis=0)

    ig_values = (input_batch - baseline) * avg_grads

    return ig_values.numpy()

def compute_global_importance(model, full_dataset, target_idx=0, baseline=None, batch_size=512):
    all_ig_values = []
    print(f"Analyzing Target Index: {target_idx}...")

    for i in tqdm(range(0, len(full_dataset), batch_size)):
        batch = full_dataset[i : i + batch_size]
        batch_ig = get_integrated_gradients_batch(model, batch, target_idx, baseline=baseline)
        all_ig_values.append(batch_ig)

    all_ig_values = np.concatenate(all_ig_values, axis=0)

    return all_ig_values


def plot_ig_summary(ig_values, feature_values, feature_names, name, top_k=None):
    mean_abs_ig = np.mean(np.abs(ig_values), axis=0)
    sorted_indices = np.argsort(mean_abs_ig)

    if top_k is not None:
        sorted_indices = sorted_indices[-top_k:]

    num_features = len(sorted_indices)

    plt.figure(figsize=(10, 0.5 * num_features + 2))
    plt.ylim(-1, num_features)
    plt.xlim(np.min(ig_values)*1.1, np.max(ig_values)*1.1)
    plt.axvline(0, color="#dddddd", zorder=-1)
    cmap = plt.get_cmap('coolwarm')

    for i, idx in enumerate(sorted_indices):
        current_igs = ig_values[:, idx]
        current_vals = feature_values.loc[:, feature_names[idx]]

        val_min, val_max = np.min(current_vals), np.max(current_vals)
        if val_max - val_min == 0:
            norm_vals = np.zeros_like(current_vals) + 0.5 # 값이 다 같으면 회색
        else:
            norm_vals = (current_vals - val_min) / (val_max - val_min)

        noise = np.random.normal(0, 0.12, size=len(current_igs))
        y_positions = i + noise

        plt.scatter(current_igs, y_positions, c=norm_vals, cmap=cmap,s=16, alpha=0.3, edgecolors='none')


    plt.yticks(range(num_features), [feature_names[i] for i in sorted_indices], fontsize=12)
    plt.xlabel("IG Value (Impact on Model Output)", fontsize=12)
    plt.title("Integrated Gradients Summary Plot {}".format(name), fontsize=14, fontweight='bold')

    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['left'].set_visible(False)
    plt.tick_params(axis='y', length=0)

    norm = mcolors.Normalize(vmin=0, vmax=1)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    cbar = plt.colorbar(sm, ax=plt.gca(), ticks=[0, 1], aspect=40)
    cbar.ax.set_yticklabels(['Low', 'High'], fontsize=10)
    cbar.set_label('Feature Value', size=12)

    plt.tight_layout()
    plt.show()

    return

In [ ]:
mean_val = np.mean(data_transform[feature], axis=0)
baseline_mean = tf.convert_to_tensor(mean_val, dtype=tf.float32)

global_imp_1 = compute_global_importance(best_model, data_transform[feature], target_idx=0, baseline=baseline_mean)
global_imp_2 = compute_global_importance(best_model, data_transform[feature], target_idx=1, baseline=baseline_mean)
global_imp_3 = compute_global_importance(best_model, data_transform[feature], target_idx=2, baseline=baseline_mean)

plot_ig_summary(global_imp_1, data_transform[feature], feature, name='Max.temp.', top_k=5)
plot_ig_summary(global_imp_2, data_transform[feature], feature, name='SoH', top_k=5)
plot_ig_summary(global_imp_3, data_transform[feature], feature, name='DCIR increase rate', top_k=5)

# Cosine similarity calculation

In [ ]:
def calculate_cosine_similarity(grad1_list, grad2_list):
    G1_flat = tf.concat([tf.reshape(g, [-1]) if g is not None else tf.zeros_like(tf.reshape(w, [-1])) for g, w in zip(grad1_list, W_shared_list)], axis=0)
    G2_flat = tf.concat([tf.reshape(g, [-1]) if g is not None else tf.zeros_like(tf.reshape(w, [-1])) for g, w in zip(grad2_list, W_shared_list)], axis=0)

    dot_product = tf.reduce_sum(G1_flat * G2_flat)
    norm_G1 = tf.norm(G1_flat)
    norm_G2 = tf.norm(G2_flat)

    if norm_G1 == 0.0 or norm_G2 == 0.0:
        return 0.0

    cosine_sim = dot_product / (norm_G1 * norm_G2)
    return cosine_sim.numpy()

optimizer = tf.keras.optimizers.Adam()

model = keras.Model(inputs=input_layer, outputs=[output_y1, output_y2, output_y3])
model.compile(
    optimizer='adam',
    loss={'output_y1': 'mean_squared_error', 'output_y2': 'mean_squared_error', 'output_y3': 'mean_squared_error'},
    loss_weights={'output_y1': 100,'output_y3': 10, 'output_y4': 1})

similarity_history_12, similarity_history_13, similarity_history_23 = [], [], []

shared_layer_names = [i.name for i in list(best_model.layers) if 'dense' in i.name]

W_shared_list = []
for layer_name in shared_layer_names:
    layer = model.get_layer(layer_name)
    W_shared_list.extend(layer.trainable_weights)

for epoch in range(300):
    epoch_avg_sim_12, epoch_avg_sim_13, epoch_avg_sim_23 = [], [], []
    for (x_batch, y_batch) in train_dataset:
        with tf.GradientTape(persistent=True) as tape:
            predictions = model(x_batch, training=True)

            loss1 = tf.reduce_mean(tf.square(tf.cast(y_batch[0], tf.float32) - predictions[0]), name='loss_1')
            loss2 = tf.reduce_mean(tf.square(tf.cast(y_batch[1], tf.float32) - predictions[1]), name='loss_2')
            loss3 = tf.reduce_mean(tf.square(tf.cast(y_batch[2], tf.float32) - predictions[2]), name='loss_3')
            total_loss = (loss1 * 100) + (loss2 * 10) + (loss3 * 1)

        G1 = tape.gradient(loss1, W_shared_list)
        G2 = tape.gradient(loss2, W_shared_list)
        G3 = tape.gradient(loss3, W_shared_list)

        sim_1_2 = calculate_cosine_similarity(G1, G2)
        sim_1_3 = calculate_cosine_similarity(G1, G3)
        sim_2_3 = calculate_cosine_similarity(G2, G3)

        epoch_avg_sim_12.append(sim_1_2)
        epoch_avg_sim_13.append(sim_1_3)
        epoch_avg_sim_23.append(sim_2_3)

        total_grads = tape.gradient(total_loss, model.trainable_variables)
        optimizer.apply_gradients(zip(total_grads, model.trainable_variables))

    similarity_history_12.append(np.mean(epoch_avg_sim_12))
    similarity_history_13.append(np.mean(epoch_avg_sim_13))
    similarity_history_23.append(np.mean(epoch_avg_sim_23))

plt.plot(range(len(similarity_history_12)), similarity_history_12, label='Max.temp.-SoH')
plt.plot(range(len(similarity_history_12)), similarity_history_13, label='Max.temp.-DCIR increase rate')
plt.plot(range(len(similarity_history_12)), similarity_history_23, label='SoH-DCIR increase rate')
plt.xlabel('Cosine similarity', fontsize=13)
plt.ylabel('Epoch', fontsize=13)
plt.legend()
plt.show()

# T-SEN visualization

In [ ]:
tsne_feature_1 = ['2.0 C_dischar_pulse_skew', '1.65 C_dischar_rest_kurt', '1.65 C_dischar_rest_R', '2.0 C_char_pulse_R']
tsne_feature_2 = ['2.0 C_dischar_pulse_kurt', '2.0 C_pulse_hysteresis']
tsne_feature_3 = ['2.0 C_char_rest_R']

tsne_feature = tsne_feature_1 + tsne_feature_2 + tsne_feature_3
tsne = TSNE(n_components=2, perplexity=15, max_iter=1500)

X_tsne = tsne.fit_transform(data_transform[tsne_feature])
tsne_df = pd.DataFrame(data=X_tsne, columns=['tSNE-1', 'tSNE-2'])

for i, j in zip(['SoH', 'temp', 'DCIR'], ['SoH', 'Max. temp.', 'DCIR increase rate']):
    plt.figure(figsize=(6, 5))
    plt.title(j, fontsize=15)
    if i == 'SoH':
        im = plt.scatter(tsne_df['tSNE-1'], tsne_df['tSNE-2'], c=df[i], s=10, alpha=0.3, vmin=60)
    elif i == 'temp':
        im = plt.scatter(tsne_df['tSNE-1'], tsne_df['tSNE-2'], c=df[i], s=10, alpha=0.3, vmin=28, vmax=35)
    else:
        im = plt.scatter(tsne_df['tSNE-1'], tsne_df['tSNE-2'], c=df[i], s=10, alpha=0.3)

    plt.xlabel('t-SNE 1', fontsize=13)
    plt.ylabel('t-SNE 2', fontsize=13)

    cbar = plt.colorbar(im)
    cbar.set_label(label=j, size=13, rotation=270, labelpad=15)

    plt.show()